# Embryo AI Pipeline: All-In-One Verification Notebook (Fixed)

This unified notebook verifies the end-to-end integration of the Embryo AI pipeline, enforcing strict framework separation, accurate threshold-based soft-gating, and robust assertion checks.

## Step 1 - Setup & Imports

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as tf_preprocess

import torch
import torch.nn as nn
from torchvision import models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using PyTorch device: {device}")
print(f"TensorFlow Version: {tf.__version__}")

## Step 2 - Load All Models

In [ ]:
# 1. Embryo Validator (TF - MobileNetV2)
validator_path = "embryo_validator_model.h5"
if os.path.exists(validator_path):
    validator_model = tf.keras.models.load_model(validator_path)
    print("Loaded Validator Model.")
else:
    print(f"Warning: {validator_path} not found.")
    validator_model = None

# 2. Stage Classifier (TF - MobileNetV2)
classifier_path = "embryo_model_turbo.h5"
if os.path.exists(classifier_path):
    classifier_model = tf.keras.models.load_model(classifier_path)
    print("Loaded Stage Classifier Model.")
else:
    print(f"Warning: {classifier_path} not found.")
    classifier_model = None

# 3. Grading Model (PyTorch - EfficientNet-B0)
class MultiHeadEfficientNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.base = models.efficientnet_b0(weights='DEFAULT')
        self.base.classifier = nn.Identity()
        self.h_exp = nn.Linear(1280, 5)
        self.h_icm = nn.Linear(1280, 4)
        self.h_te = nn.Linear(1280, 4)
    def forward(self, x):
        f = torch.flatten(self.base(x), 1)
        return self.h_exp(f), self.h_icm(f), self.h_te(f)

grading_path = "embryo_grading_v4.pth"
grading_model = MultiHeadEfficientNet().to(device)
if os.path.exists(grading_path):
    grading_model.load_state_dict(torch.load(grading_path, map_location=device))
    print("Loaded Grading Model.")
else:
    print(f"Warning: {grading_path} not found.")

grading_model.eval()

# Preprocessing Definitions (Kept completely separate as requested)
grading_transform = transforms.Compose([
    transforms.Resize((224, 224)), # Explicit 224x224 resize
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## Step 3 - Define Pipeline Function

In [ ]:
def run_pipeline(image_path):
    # --- ERROR HANDLING: Safe image load ---
    try:
        img = Image.open(image_path).convert('RGB')
    except Exception as e:
        return {
            "validation_prob": None,
            "validation_status": "error",
            "stage": None,
            "grading": None,
            "confidence": None,
            "error_msg": str(e)
        }
        
    # --- 1. VALIDATION (MobileNetV2: 160x160) ---
    stage_img = img.resize((160, 160))
    val_array = tf_preprocess(np.expand_dims(np.array(stage_img).astype('float32'), axis=0))
    
    if validator_model:
        prob = float(validator_model.predict(val_array, verbose=0)[0][0])
    else:
        prob = 1.0 # fallback for dev
        
    # Soft-Gate Logic
    if prob < 0.60:
        val_status = "reject"
    elif 0.60 <= prob < 0.85:
        val_status = "warn"
    else:
        val_status = "accept"
        
    # HARD STOP on Reject
    if val_status == "reject":
        return {
            "validation_prob": prob,
            "validation_status": val_status,
            "stage": None,
            "grading": None,
            "confidence": None
        }
        
    # --- 2. STAGE CLASSIFICATION (MobileNetV2: 160x160) ---
    # We reuse val_array because it shares exact 160x160 + tf_preprocess reqs with validation
    if classifier_model:
        preds = classifier_model.predict(val_array, verbose=0)[0]
    else:
        preds = [0, 0, 0, 0, 1] # fallback to blastocyst
        
    class_names = ['2-cell', '4-cell', '8-cell', 'morula', 'blastocyst']
    idx = int(np.argmax(preds))
    stage_label = class_names[idx]
    stage_conf = float(preds[idx])
    
    # HARD STOP on Non-Blastocyst (skip grading)
    if stage_label != "blastocyst":
        return {
            "validation_prob": prob,
            "validation_status": val_status,
            "stage": stage_label,
            "grading": None,
            "confidence": stage_conf
        }

    # --- 3. GRADING (EfficientNet-B0: 224x224) ---
    # Separate explicit pipeline branch with PyTorch logic
    grading_tensor = grading_transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        if grading_model:
            p_exp, p_icm, p_te = grading_model(grading_tensor)
            e_idx = int(torch.argmax(p_exp, 1).item())
            i_idx = int(torch.argmax(p_icm, 1).item())
            t_idx = int(torch.argmax(p_te, 1).item())
        else:
            e_idx, i_idx, t_idx = 0, 0, 0
            
    exp_map = {0: 1, 1: 2, 2: 3, 3: 4, 4: 5}
    icm_te_map = {0: 'A', 1: 'B', 2: 'C', 3: 'NA'}
    
    grading_tuple = (exp_map[e_idx], icm_te_map[i_idx], icm_te_map[t_idx])
    
    return {
        "validation_prob": prob,
        "validation_status": val_status,
        "stage": stage_label,
        "grading": grading_tuple,
        "confidence": stage_conf
    }

## Step 4 - Prepare Test Cases

In [ ]:
os.makedirs("all_in_one_tests", exist_ok=True)

# Synthetics
cv2.imwrite("all_in_one_tests/blank.jpg", np.zeros((160, 160, 3), dtype=np.uint8))
cv2.imwrite("all_in_one_tests/noisy.jpg", np.random.randint(0, 256, (160, 160, 3), dtype=np.uint8))

test_cases = [
    {"id": "Valid Blastocyst", "path": "2b.jpeg"},                # Real image
    {"id": "Invalid Image (Logo)", "path": "embryo_ai_logo_1776785134392.png"},
    {"id": "Screenshot / UI", "path": "embryo_ai_logo_1776785134392.png"},   # Using logo as proxy for UI
    {"id": "Random Photo (Not found)", "path": "some_missing_file.jpg"},      # Will test error handling
    {"id": "Blank Image", "path": "all_in_one_tests/blank.jpg"},
    {"id": "Noisy Image", "path": "all_in_one_tests/noisy.jpg"}
]
print("Test cases prepared.")

## Step 5 - Run Pipeline on All Tests

In [ ]:
results_list = []
for case in test_cases:
    print(f"Executing: {case['id']}")
    res = run_pipeline(case['path'])
    res['id'] = case['id']
    res['path'] = case['path']
    results_list.append(res)
print("Execution complete.")

## Step 6 - Display Results & Summary Table

In [ ]:
# Visual display of inputs and outcomes
plt.figure(figsize=(16, 8))
for i, res in enumerate(results_list):
    plt.subplot(2, 3, i + 1)
    
    if res.get('validation_status') == 'error':
        plt.text(0.5, 0.5, f"[LOAD ERROR]\n{res.get('error_msg', '')}", ha='center', color='red')
        plt.title(res['id'], color='red')
        plt.axis('off')
        continue
        
    try:
        img = Image.open(res['path'])
        plt.imshow(img)
    except:
        pass
        
    grade_str = str(res['grading']) if res['grading'] else 'N/A'
    color = 'green' if res['validation_status'] == 'accept' else 'red'
    if res['validation_status'] == 'warn': color = 'orange'
        
    title = f"{res['id']}\nVal: {res['validation_status'].upper()} ({res['validation_prob']:.2f})\n"
    title += f"Stage: {res['stage']} | Grade: {grade_str}"
    
    plt.title(title, color=color)
    plt.axis('off')
    
plt.tight_layout()
plt.show()

# Summary Table
df = pd.DataFrame(results_list)
df['Image'] = df['id']
df['Validation'] = df.apply(lambda r: f"{r['validation_status']} ({r.get('validation_prob')})" if pd.notnull(r.get('validation_prob')) else r.get('validation_status'), axis=1)
df['Stage'] = df['stage'].fillna('None')
df['Grading'] = df['grading'].apply(lambda x: str(x) if x else 'None')
df['Status'] = df['validation_status']

summary_df = df[['Image', 'Validation', 'Stage', 'Grading', 'Status']]
display(summary_df.style.applymap(lambda x: 'background-color: #ffcccc; color: #000' if x=='reject' or x=='error' else ('background-color: #ffffcc; color: #000' if x=='warn' else ''), subset=['Status']))

## Step 7 - Assertions (CRITICAL)

In [ ]:
failed_assertions = 0
print("--- Running Verification Assertions ---")

for res in results_list:
    # Skip assertion checks if it's a file error (handled natively)
    if res['validation_status'] == 'error':
        continue
        
    # 1. If rejected: assert stage == None AND grading == None
    if res['validation_status'] == 'reject':
        if res['stage'] is not None or res['grading'] is not None:
            print(f"❌ [FAIL] {res['id']} was rejected but processed downstream!")
            failed_assertions += 1
            
    # 2. If not blastocyst: assert grading == None
    if res['stage'] != 'blastocyst':
        if res['grading'] is not None:
            print(f"❌ [FAIL] {res['id']} is not a blastocyst but executed grading!")
            failed_assertions += 1

    # 3. If blastocyst: assert grading executed
    if res['validation_status'] in ['accept', 'warn'] and res['stage'] == 'blastocyst':
        if res['grading'] is None:
            print(f"❌ [FAIL] {res['id']} is a valid blastocyst but skipped grading!")
            failed_assertions += 1

if failed_assertions == 0:
    print("✅ All pipeline state assertions passed.")
else:
    print(f"❌ {failed_assertions} assertions failed!")

## Step 8 - Final Summary

In [ ]:
total = len(results_list)
passed = total - failed_assertions

print("=========================================")
print("      PIPELINE EXECUTION SUMMARY         ")
print("=========================================")
print(f"Total Tests Run: {total}")
print(f"Tests Passed:    {passed}")
print(f"Tests Failed:    {failed_assertions}")
print("=========================================")
if failed_assertions == 0:
    print("✅ SUCCESS: The pipeline strictly obeys validation gating,")
    print("           prevents confident predictions on invalid inputs,")
    print("           and confines grading to valid blastocysts.")
else:
    print("❌ FAILURE: Review pipeline state leaks or assertion errors.")
print("=========================================")